In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("dataset/space.csv")

In [3]:
df_model = df[["pl_orbper","pl_orbsmax" ,"st_mass" ,"pl_orbeccen" ,"st_rad" ,"pl_rade"]]

In [4]:
df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 34993 entries, 0 to 34992
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   pl_orbper    31931 non-null  float64
 1   pl_orbsmax   19331 non-null  float64
 2   st_mass      29557 non-null  float64
 3   pl_orbeccen  17539 non-null  float64
 4   st_rad       32413 non-null  float64
 5   pl_rade      24119 non-null  float64
dtypes: float64(6)
memory usage: 1.6 MB


In [5]:
print(df_model.isnull().sum())

pl_orbper       3062
pl_orbsmax     15662
st_mass         5436
pl_orbeccen    17454
st_rad          2580
pl_rade        10874
dtype: int64


In [6]:
df_model = df_model.dropna(subset = "pl_orbper")

In [7]:
df_model = df_model.fillna(df_model.median())

In [8]:
X = df_model[["pl_orbsmax", "st_mass", "pl_orbeccen", "st_rad", "pl_rade"]].values
y = df_model["pl_orbper"].values
y = np.log1p(y)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)


In [10]:
from sklearn.preprocessing import RobustScaler

# Burada data değerlerim büyük ve MSELoss kullandığım için değerler aşırı büyük çıkıyor bu yüzden ölçeklendirme lazım.
# Ancak StandartScaler kullanmak için verinin numpy dizisi olması gerekiyor. Bu sebeple numpy ise devam ediyoruz değilse numpy çevirip CPU'ya alıyoruz.
# Ölçeklendirip geri GPU'ya yolluyoruz.
X_train_np = X_train if isinstance(X_train, np.ndarray) else X_train.cpu().numpy()
y_train_np = y_train if isinstance(y_train, np.ndarray) else y_train.cpu().numpy()
X_test_np = X_test if isinstance(X_test, np.ndarray) else X_test.cpu().numpy()
y_test_np = y_test if isinstance(y_test, np.ndarray) else y_test.cpu().numpy()

# Burada StandartScaler() sınıfından bir nesne üretiyoruz fit_transform() ve transform() 'u kullanabilmek için.
scaler_X = RobustScaler()
scaler_y = RobustScaler()

# Burada ise ürettiğimiz nesneler ile method'ları kullanıp verileri ölçeklendiriyoruz
X_train_scaled = scaler_X.fit_transform(X_train_np)
X_test_scaled = scaler_X.transform(X_test_np)

# y değerlerini 2D (N, 1) yapıp ölçeklendiriyoruz.
# Çünkü boyutları uyumlu olmak zorunda. Yoksa hata alırız.
y_train_scaled = scaler_y.fit_transform(y_train_np.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test_np.reshape(-1, 1))



# Yukarıda reshape ile zaten boyuları eşitlediğimiz için burada unsqueeze() veya squeeze() kullanmamıza gerek yok. Zaten eşitler. 
X_train = torch.tensor(data= X_train_scaled, dtype= torch.float32)
X_test = torch.tensor(data= X_test_scaled, dtype= torch.float32)

y_train = torch.tensor(data= y_train_scaled, dtype= torch.float32)
y_test = torch.tensor(data= y_test_scaled, dtype= torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Verileri GPU'ya taşıyoruz
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

# NOT!!! Scaler çıktıları her zaman 2 boyutludur(2D).

In [11]:
X_train.to(device)
X_test.to(device)

y_train.to(device)
y_test.to(device)

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [12]:
import torch.nn as nn

In [13]:
class PlanetPredict(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(in_features=5, out_features=128)
        self.layer2 = nn.Linear(in_features=128, out_features=64)
        self.layer3 = nn.Linear(in_features=64, out_features=32)
        self.layer4 = nn.Linear(in_features=32, out_features=16)
        self.layer5 = nn.Linear(in_features=16, out_features=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, X: torch.tensor) -> float:
        return self.layer5(self.relu(self.layer4(self.relu(self.layer3(self.relu(self.layer2(self.relu(self.layer1(X)))))))))

In [14]:
torch.manual_seed(123)
model = PlanetPredict()
model = model.to(device)

In [15]:
loss_fn = nn.MSELoss()

#weight_decay 
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.0008, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',      # loss düşsün istiyoruz
    factor=0.5,      # lr'yi yarıya indir
    patience=50      # 20 epoch iyileşmezse
)

In [16]:
from torchmetrics import MeanSquaredError, R2Score

loss_calculate = MeanSquaredError().to(device)

In [17]:
model = model.to(device)
train_mse = MeanSquaredError(squared=True, num_outputs=1).to(device)
train_r2 = R2Score().to(device)


test_mse = MeanSquaredError(squared=True, num_outputs=1).to(device)
test_r2 = R2Score().to(device)

In [18]:


epochs = 1000
best_loss = float('inf')  # sonsuz büyük başlangıç değeri
patience = 1000
patience_counter = 0

for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        model.train()

        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        train_r2_score = train_r2(y_pred, y_batch)
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    

    model.eval()
    with torch.inference_mode():
        test_pred = model(X_test)
        test_loss = test_mse(test_pred, y_test)
        test_r2_score = test_r2(test_pred, y_test)
    scheduler.step(test_loss)
    if test_loss < best_loss:
        best_loss = test_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'saved_model/best_model.pth')
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break
        
    if epoch % 15 == 0:
        print(f"Epoch: {epoch}, Train Loss: {loss:.4f}, Train R2 Score: {train_r2_score:.4f}, Test Loss: {test_loss:.4f}, Test R2 Score: {test_r2_score:.4f}")

    train_r2.reset()
    train_mse.reset() 
    test_r2.reset()
    test_mse.reset()
    
    

Epoch: 0, Train Loss: 0.7617, Train R2 Score: 0.1802, Test Loss: 0.6424, Test R2 Score: 0.1766
Epoch: 15, Train Loss: 0.3793, Train R2 Score: 0.5157, Test Loss: 0.3847, Test R2 Score: 0.5069
Epoch: 30, Train Loss: 0.4708, Train R2 Score: 0.3397, Test Loss: 0.4866, Test R2 Score: 0.3764
Epoch: 45, Train Loss: 0.3046, Train R2 Score: 0.5237, Test Loss: 0.3599, Test R2 Score: 0.5387
Epoch: 60, Train Loss: 0.2486, Train R2 Score: 0.6806, Test Loss: 0.2929, Test R2 Score: 0.6246
Epoch: 75, Train Loss: 0.3282, Train R2 Score: 0.6427, Test Loss: 0.2430, Test R2 Score: 0.6885
Epoch: 90, Train Loss: 0.1806, Train R2 Score: 0.7329, Test Loss: 0.2284, Test R2 Score: 0.7072
Epoch: 105, Train Loss: 0.2068, Train R2 Score: 0.7550, Test Loss: 0.2096, Test R2 Score: 0.7314
Epoch: 120, Train Loss: 0.1659, Train R2 Score: 0.8277, Test Loss: 0.2105, Test R2 Score: 0.7302
Epoch: 135, Train Loss: 0.2501, Train R2 Score: 0.6799, Test Loss: 0.2025, Test R2 Score: 0.7405
Epoch: 150, Train Loss: 0.1996, Train 

In [19]:
print(type(X_train))
print(type(y_train))
print(y_train_scaled.min(), y_train_scaled.max())

<class 'torch.Tensor'>
<class 'torch.Tensor'>
-1.4366844923948892 10.605022336806126


In [20]:
print(type(y_train_np))
print(y_train_np.min(), y_train_np.max())

<class 'numpy.ndarray'>
0.0868254588745947 19.811962649070857


In [21]:
print(df_model["pl_orbper"].describe())

count    3.193100e+04
mean     1.366676e+04
std      2.250981e+06
min      9.070629e-02
25%      4.458157e+00
50%      1.050683e+01
75%      2.732179e+01
max      4.020000e+08
Name: pl_orbper, dtype: float64
